In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("processed_data.csv")

In [3]:
df

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,-0.250000,0.750000,6.250000,NaN,1.0
1,AS14.01,2014-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,0.333333,6.333333,NaN,NaN
2,AS14.01,2014-03-21,0.134050,14.269173,55.095526,53.024000,6.193000,NaN,57.402000,29.968875,...,70.418846,NaN,23.029000,NaN,1.0,0.200000,0.200000,6.200000,153.665872,NaN
3,AS14.01,2014-03-22,0.236880,7.781160,51.697063,46.662000,7.025333,NaN,NaN,14.020429,...,37.305000,NaN,39.207000,NaN,1.0,0.600000,0.500000,6.400000,47.613651,1.0
4,AS14.01,2014-03-23,0.142741,12.134396,53.442031,31.448667,10.850750,NaN,NaN,12.137167,...,NaN,NaN,30.086000,30.386,NaN,0.200000,0.800000,6.800000,67.067644,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,AS14.33,2014-05-27,0.012704,8.545776,59.382943,62.347000,NaN,NaN,NaN,17.835750,...,NaN,NaN,18.724333,NaN,1.0,-0.600000,0.400000,6.200000,113.607750,1.0
1264,AS14.33,2014-05-28,0.103301,9.590398,37.238750,43.907571,NaN,NaN,44.738625,27.891895,...,NaN,NaN,5.111000,NaN,1.0,0.000000,1.200000,8.200000,113.657683,1.0
1265,AS14.33,2014-05-29,0.169354,6.452161,54.388125,68.369000,NaN,NaN,NaN,9.734000,...,0.939000,NaN,1.599500,NaN,1.0,-1.333333,1.000000,7.000000,118.978033,1.0
1266,AS14.33,2014-05-30,0.192901,14.776845,46.412278,37.174286,NaN,NaN,NaN,10.522000,...,37.594571,2.690667,15.521667,NaN,1.0,-0.800000,-0.400000,6.800000,86.342245,NaN


In [4]:
df.astype({"id": "string", "date": "datetime64[ns]", "sms":"Int8", "call":"Int8"}).dtypes

id                      string[python]
date                    datetime64[ns]
activity                       float64
appCat.builtin                 float64
appCat.communication           float64
appCat.entertainment           float64
appCat.finance                 float64
appCat.game                    float64
appCat.office                  float64
appCat.other                   float64
appCat.social                  float64
appCat.travel                  float64
appCat.unknown                 float64
appCat.utilities               float64
appCat.weather                 float64
call                              Int8
circumplex.arousal             float64
circumplex.valence             float64
mood                           float64
screen                         float64
sms                               Int8
dtype: object

# Outlier Removal

### Incorrect Value Removal

In [5]:
# mood --> 1-10
# circumplex.arousal --> -2 to 2
# circumplex.valence --> -2 to 2
# activity --> 0-1
# screen --> time (Has to be positive)
# call --> 0 or 1
# sms --> 0 or 1
# appCat.builtin --> time (Has to be positive)
# appCat.communication --> time (Has to be positive)
# appCat.entertainment --> time (Has to be positive)
# appCat.finance --> time (Has to be positive)
# appCat.game --> time (Has to be positive)
# appCat.office --> time (Has to be positive)
# appCat.other --> time (Has to be positive)
# appCat.social --> time (Has to be positive)
# appCat.travel --> time (Has to be positive)
# appCat.unknown --> time (Has to be positive)
# appCat.utilities --> time (Has to be positive)
# appCat.weather --> time (Has to be positive)

In [6]:
constraints = {
    'mood': (1, 10),
    'circumplex.arousal': (-2, 2),
    'circumplex.valence': (-2, 2),
    'activity': (0, 1)
}

In [7]:
for col, (low, high) in constraints.items():
    if not df[col].between(low, high, inclusive="both").any():
        print(f"{col} has violations")

In [8]:
# No Violation in Mood, circumplex.arousal, circumplex.valence, activity

In [9]:
positive_cols = [
    'screen',
    'appCat.builtin', 'appCat.communication', 'appCat.entertainment',
    'appCat.finance', 'appCat.game', 'appCat.office', 'appCat.other',
    'appCat.social', 'appCat.travel', 'appCat.unknown',
    'appCat.utilities', 'appCat.weather'
]

for col in positive_cols:
    mask = df[col] < 0
    
    if mask.any():
        print(f"\nBefore fixing {col}:")
        print(df.loc[mask, [col]])

        df.loc[mask, col] = np.nan

        print(f"After fixing {col}:")
        print(df.loc[mask, [col]])


Before fixing appCat.builtin:
     appCat.builtin
275     -772.768558
After fixing appCat.builtin:
     appCat.builtin
275             NaN


In [10]:
binary_cols = ['call', 'sms']

for bincol in binary_cols:
    mask = (~df[bincol].isin([0, 1])) & (df[bincol].notna())
    print(df.loc[mask, ["id", "date", bincol]])

Empty DataFrame
Columns: [id, date, call]
Index: []
Empty DataFrame
Columns: [id, date, sms]
Index: []


In [11]:
df.groupby(df.columns.tolist()).size().reset_index(name='count').query('count > 1')

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms,count


# Time Aware Outlier Detection

### Winsorizer for Screen with percentile 95

In [12]:
df = df.sort_values(["id", "date"])
new_df = df.copy()

In [13]:
def winsorize_p95(x):
    cap = x.quantile(0.95)
    return np.where(x > cap, cap, x)

In [14]:
cols_winsor = ["screen"]

for col in cols_winsor:
    new_df[col] = df.groupby("id")[col].transform(winsorize_p95)

### Log1p for all appCat columns

In [15]:
app_cols = [
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
    "appCat.social", "appCat.travel", "appCat.unknown",
    "appCat.utilities", "appCat.weather"
]

for col in app_cols:
    new_df[col] = np.log1p(df[col])

In [16]:
new_df

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,-0.250000,0.750000,6.250000,NaN,1.0
1,AS14.01,2014-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,0.333333,6.333333,NaN,NaN
2,AS14.01,2014-03-21,0.134050,2.725836,4.027056,3.989428,1.973108,NaN,4.067350,3.432983,...,4.268562,NaN,3.179261,NaN,1.0,0.200000,0.200000,6.200000,153.665872,NaN
3,AS14.01,2014-03-22,0.236880,2.172608,3.964560,3.864134,2.082603,NaN,NaN,2.709411,...,3.645580,NaN,3.694041,NaN,1.0,0.600000,0.500000,6.400000,47.613651,1.0
4,AS14.01,2014-03-23,0.142741,2.575234,3.997136,3.479659,2.472391,NaN,NaN,2.575445,...,NaN,NaN,3.436758,3.446362,NaN,0.200000,0.800000,6.800000,67.067644,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,AS14.33,2014-05-27,0.012704,2.256099,4.100707,4.148628,NaN,NaN,NaN,2.935757,...,NaN,NaN,2.981853,NaN,1.0,-0.600000,0.400000,6.200000,113.607750,1.0
1264,AS14.33,2014-05-28,0.103301,2.359948,3.643849,3.804606,NaN,NaN,3.822943,3.363561,...,NaN,NaN,1.810090,NaN,1.0,0.000000,1.200000,8.200000,113.657683,1.0
1265,AS14.33,2014-05-29,0.169354,2.008504,4.014365,4.239440,NaN,NaN,NaN,2.373416,...,0.662172,NaN,0.955319,NaN,1.0,-1.333333,1.000000,7.000000,118.978033,1.0
1266,AS14.33,2014-05-30,0.192901,2.758543,3.858881,3.642162,NaN,NaN,NaN,2.444258,...,3.653112,1.305807,2.804673,NaN,1.0,-0.800000,-0.400000,6.800000,86.342245,NaN


In [17]:
range_df = new_df[app_cols].agg(['min', 'max']).T.reset_index()
range_df.columns = ['variable', 'min', 'max']
print(range_df)

                variable       min       max
0         appCat.builtin  0.649195  7.244267
1   appCat.communication  0.739554  6.260953
2   appCat.entertainment  0.700123  7.409587
3         appCat.finance  0.695145  5.011841
4            appCat.game  0.698135  7.849288
5          appCat.office  0.043059  7.310459
6           appCat.other  1.509507  6.584449
7          appCat.social  0.545227  8.232221
8          appCat.travel  0.076961  7.472512
9         appCat.unknown  0.105261  7.714649
10      appCat.utilities  0.694147  5.898332
11        appCat.weather  0.696641  5.217479


# Data Change Info

In [18]:
features = df.select_dtypes(include=['float64', 'Int8']).columns
df_before = df 
df_after = new_df

In [19]:
before_stats = df_before[features].describe()
after_stats = df_after[features].describe()

diff = after_stats - before_stats

print(diff)

       activity  appCat.builtin  appCat.communication  appCat.entertainment  \
count       0.0        0.000000              0.000000              0.000000   
mean        0.0      -23.267457            -41.730244            -91.334327   
std         0.0      -75.984372            -36.394971           -148.218143   
min         0.0       -0.264805             -0.355446             -0.313877   
25%         0.0       -5.766940            -23.871641            -13.166524   
50%         0.0       -9.836451            -34.916083            -44.348845   
75%         0.0      -18.159092            -48.367794           -106.891473   
max         0.0    -1391.810448           -516.457047          -1643.334413   

       appCat.finance  appCat.game  appCat.office  appCat.other  \
count        0.000000     0.000000       0.000000      0.000000   
mean       -22.823024  -152.625122     -39.285460    -18.820681   
std        -23.900488  -257.207613    -104.525203    -40.095375   
min         -0.30885

# NaN Handling

### NaN count analysis

In [20]:
pd.set_option('display.max_columns', None)

In [21]:
nan_counts = df.isna().groupby(df['id']).sum()
nan_counts.head(30)

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
id,,,,,,,,,,,,,,,,,,,,,
AS14.01,0,0,2,2,2,2,15,40,39,2,2,22,42,4,46,15,0,0,0,2,33
AS14.02,0,0,5,7,8,11,41,41,41,11,9,29,41,37,41,2,0,0,0,6,17
AS14.03,0,0,1,0,0,0,22,47,20,0,43,34,15,26,45,15,0,0,0,0,28
AS14.05,0,0,3,2,0,0,53,53,52,0,19,52,15,11,53,26,0,0,0,0,37
AS14.06,0,0,0,2,0,0,45,37,21,1,0,14,44,45,36,13,0,0,0,1,36
AS14.07,0,0,7,2,1,4,48,47,48,1,1,26,31,41,48,19,0,0,0,1,22
AS14.08,0,0,12,10,8,31,62,62,56,8,8,58,54,31,62,55,0,0,0,8,46
AS14.09,0,0,10,10,9,9,25,44,31,11,20,24,34,25,36,23,0,0,0,9,34
AS14.12,0,0,1,1,1,1,22,32,21,1,1,10,40,21,38,9,0,0,0,1,32


In [22]:
nan_per_row = df.isna().sum(axis=1)
df_with_nan_count = df.copy()
df_with_nan_count['nan_count'] = nan_per_row

rows_with_high_nan = df_with_nan_count[df_with_nan_count['nan_count'] >= 13]

In [23]:
rows_with_high_nan

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms,nan_count
0,AS14.01,2014-02-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-0.250000,0.750000,6.250000,NaN,1.0,14
1,AS14.01,2014-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.333333,6.333333,NaN,NaN,16
80,AS14.02,2014-04-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.000000,0.000000,6.500000,NaN,NaN,15
82,AS14.02,2014-04-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-0.666667,0.000000,5.000000,NaN,1.0,14
83,AS14.02,2014-04-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-0.500000,-0.500000,5.500000,NaN,NaN,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,AS14.31,2014-05-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,7.000000,NaN,NaN,16
1185,AS14.31,2014-05-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,7.000000,NaN,NaN,16
1186,AS14.31,2014-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,7.000000,NaN,NaN,16
1187,AS14.31,2014-05-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,1.000000,7.000000,NaN,NaN,16


In [24]:
rows_with_high_nan['id'].value_counts()

id
AS14.26    22
AS14.15    15
AS14.17    11
AS14.09     9
AS14.20     9
AS14.31     8
AS14.08     8
AS14.02     6
AS14.16     2
AS14.01     2
AS14.14     1
AS14.19     1
AS14.13     1
AS14.12     1
AS14.27     1
AS14.28     1
AS14.29     1
AS14.30     1
AS14.07     1
AS14.33     1
Name: count, dtype: int64

### Missing Days Analysis

In [25]:
missing_days = {}

for pid, group in df.groupby('id'):
    full_range = pd.date_range(start=group['date'].min(), end=group['date'].max())
    actual_dates = set(group['date'])
    
    missing = sorted(set(full_range) - actual_dates)
    
    if missing:
        missing_days[pid] = missing

missing_days

{'AS14.01': [Timestamp('2014-02-26 00:00:00'),
  Timestamp('2014-02-27 00:00:00'),
  Timestamp('2014-02-28 00:00:00'),
  Timestamp('2014-03-01 00:00:00'),
  Timestamp('2014-03-02 00:00:00'),
  Timestamp('2014-03-03 00:00:00'),
  Timestamp('2014-03-04 00:00:00'),
  Timestamp('2014-03-05 00:00:00'),
  Timestamp('2014-03-06 00:00:00'),
  Timestamp('2014-03-07 00:00:00'),
  Timestamp('2014-03-08 00:00:00'),
  Timestamp('2014-03-09 00:00:00'),
  Timestamp('2014-03-10 00:00:00'),
  Timestamp('2014-03-11 00:00:00'),
  Timestamp('2014-03-12 00:00:00'),
  Timestamp('2014-03-13 00:00:00'),
  Timestamp('2014-03-14 00:00:00'),
  Timestamp('2014-03-15 00:00:00'),
  Timestamp('2014-03-16 00:00:00'),
  Timestamp('2014-03-17 00:00:00'),
  Timestamp('2014-03-18 00:00:00'),
  Timestamp('2014-03-19 00:00:00'),
  Timestamp('2014-03-20 00:00:00'),
  Timestamp('2014-03-21 00:00:00'),
  Timestamp('2014-03-22 00:00:00'),
  Timestamp('2014-03-23 00:00:00'),
  Timestamp('2014-03-24 00:00:00'),
  Timestamp('2014

In [26]:
df['date'] = pd.to_datetime(df['date'])
df['prev_date'] = df.groupby('id')['date'].shift(1)
df['gap'] = (df['date'] - df['prev_date']).dt.days

fixable_missing = df[df['gap'] <= 4].copy()
fixable_missing[['id', 'prev_date', 'date']]

,id,prev_date,date
1,AS14.01,2014-02-26,2014-02-27
3,AS14.01,2014-03-21,2014-03-22
4,AS14.01,2014-03-22,2014-03-23
5,AS14.01,2014-03-23,2014-03-24
6,AS14.01,2014-03-24,2014-03-25
...,...,...,...
1263,AS14.33,2014-05-26,2014-05-27
1264,AS14.33,2014-05-27,2014-05-28
1265,AS14.33,2014-05-28,2014-05-29
1266,AS14.33,2014-05-29,2014-05-30


In [27]:
not_fixable_missing = df[df['gap'] > 4].copy()
not_fixable_missing[['id', 'prev_date', 'date']]

,id,prev_date,date
2,AS14.01,2014-02-27,2014-03-21
390,AS14.12,2014-03-15,2014-03-27
646,AS14.17,2014-03-14,2014-03-21


### NaN and Missing Days Handling

In [28]:
df = new_df.copy()
df = df.sort_values(['id', 'date']).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ['id', 'date']]
appcat_cols  = [c for c in feature_cols if c.startswith('appCat')]

In [29]:
df['nan_count'] = df[feature_cols].isnull().sum(axis=1)
df = df[df['nan_count'] <= 13].drop(columns='nan_count').reset_index(drop=True)
print(f"Rows after dropping high-NaN rows: {len(df)}")

Rows after dropping high-NaN rows: 1174


In [30]:
df['date'] = pd.to_datetime(df['date'])

In [31]:
all_participants = []

for pid, group in df.groupby('id'):
    group = group.sort_values('date').reset_index(drop=True)

    participant_means = group[feature_cols].mean()
    participant_modes = group[feature_cols].mode().iloc[0]

    rows_to_add = []
    diffs = group['date'].diff().dt.days.fillna(1)

    for idx in range(1, len(group)):
        gap = int(diffs[idx])

        if gap > 4:
            block_before = group.iloc[:idx]
            block_after  = group.iloc[idx:]

            def max_consecutive_run(grp):
                d = grp['date'].diff().dt.days.fillna(1)
                run = 1; best = 1; cur = 1
                for v in d[1:]:
                    cur = cur + 1 if v == 1 else 1
                    best = max(best, cur)
                return best

            valid_blocks = []
            for block in [block_before, block_after]:
                if max_consecutive_run(block) >= 10:
                    valid_blocks.append(block)

            if valid_blocks:
                group = pd.concat(valid_blocks).reset_index(drop=True)
            diffs = group['date'].diff().dt.days.fillna(1)

        elif 1 < gap <= 4:
            prev_date = group.loc[idx - 1, 'date']
            for offset in range(1, gap):
                missing_date = prev_date + pd.Timedelta(days=offset)
                new_row = {'id': pid, 'date': missing_date}

                for col in appcat_cols:
                    new_row[col] = participant_means[col]

                for col in ['sms', 'call', 'activity']:
                    new_row[col] = participant_modes[col]

                prev_row = group.iloc[idx - 1]
                next_row = group.iloc[idx]
                for col in ['circumplex.valence', 'circumplex.arousal', 'screen', 'mood']:
                    prev_val = prev_row[col]
                    next_val = next_row[col]
                    if pd.notna(prev_val) and pd.notna(next_val):
                        new_row[col] = (prev_val + next_val) / 2
                    elif pd.notna(prev_val):
                        new_row[col] = prev_val
                    elif pd.notna(next_val):
                        new_row[col] = next_val
                    else:
                        new_row[col] = participant_means[col]

                rows_to_add.append(new_row)

    if rows_to_add:
        group = pd.concat([group, pd.DataFrame(rows_to_add)], ignore_index=True)
        group = group.sort_values('date').reset_index(drop=True)

    all_participants.append(group)

In [32]:
df = pd.concat(all_participants, ignore_index=True)
df = df.sort_values(['id', 'date']).reset_index(drop=True)
print(f"Rows after inserting synthetic gap rows: {len(df)}")

Rows after inserting synthetic gap rows: 1214


In [33]:
df[appcat_cols] = df[appcat_cols].fillna(0)

In [34]:
df[['sms', 'call']] = df[['sms', 'call']].fillna(0)

In [35]:
for col in ['circumplex.valence', 'circumplex.arousal']:
    ffilled = df.groupby('id')[col].transform(lambda x: x.ffill())
    bfilled = df.groupby('id')[col].transform(lambda x: x.bfill())
    df[col] = df[col].fillna((ffilled + bfilled) / 2)

In [36]:
for col in ['activity', 'screen']:
    df[col] = df.groupby('id')[col].transform(lambda x: x.fillna(x.mean()))

In [37]:
remaining_nans = df[feature_cols].isnull().sum()
print("\nRemaining NaNs per column after full cleaning:")
print(remaining_nans[remaining_nans > 0] if remaining_nans.sum() > 0 else "No NaNs")
print(f"\nFinal dataset shape: {df.shape}")


Remaining NaNs per column after full cleaning:
No NaNs

Final dataset shape: (1214, 21)


### Block of record per patient 

In [38]:
df = df.sort_values(['id', 'date'])
df['prev_date'] = df.groupby('id')['date'].shift(1)
df['gap'] = (df['date'] - df['prev_date']).dt.days

df['new_block'] = (df['gap'].isna()) | (df['gap'] > 1)
df['block_id'] = df.groupby('id')['new_block'].cumsum()
blocks = (
    df.groupby(['id', 'block_id'])
      .agg(start_date=('date', 'min'),
           end_date=('date', 'max'))
      .reset_index()
)

blocks = blocks[['id', 'start_date', 'end_date']]
blocks

,id,start_date,end_date
0,AS14.01,2014-03-21,2014-05-04
1,AS14.02,2014-03-16,2014-04-23
2,AS14.03,2014-03-21,2014-05-07
3,AS14.05,2014-03-14,2014-05-05
4,AS14.06,2014-03-24,2014-05-08
5,AS14.07,2014-03-19,2014-05-04
6,AS14.08,2014-03-13,2014-05-05
7,AS14.09,2014-03-21,2014-04-26
8,AS14.12,2014-03-27,2014-05-05
9,AS14.13,2014-03-13,2014-05-03


In [41]:
df = df.drop(['prev_date', 'gap', 'block_id', 'new_block'], axis=1)

In [42]:
df

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-03-21,0.134050,2.725836,4.027056,3.989428,1.973108,0.0,4.067350,3.432983,4.332596,4.268562,0.000000,3.179261,0.000000,1.0,0.200000,0.2,6.20,153.665872,0.0
1,AS14.01,2014-03-22,0.236880,2.172608,3.964560,3.864134,2.082603,0.0,0.000000,2.709411,3.627961,3.645580,0.000000,3.694041,0.000000,1.0,0.600000,0.5,6.40,47.613651,1.0
2,AS14.01,2014-03-23,0.142741,2.575234,3.997136,3.479659,2.472391,0.0,0.000000,2.575445,3.932739,0.000000,0.000000,3.436758,3.446362,0.0,0.200000,0.8,6.80,67.067644,0.0
3,AS14.01,2014-03-24,0.078961,1.788840,3.882423,4.813171,1.899766,0.0,1.388791,2.492627,4.330313,4.442192,0.000000,4.503868,0.000000,1.0,0.800000,0.0,6.00,76.772199,0.0
4,AS14.01,2014-03-25,0.098374,2.400335,4.254726,3.558287,2.465002,0.0,0.000000,3.427617,4.242082,0.000000,3.414537,3.147994,0.000000,0.0,0.500000,0.5,6.75,169.097283,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209,AS14.33,2014-05-26,0.115201,2.164623,3.564463,4.246735,0.000000,0.0,4.879235,2.881751,4.718911,0.000000,0.000000,0.000000,0.000000,1.0,-0.400000,0.0,5.40,300.399810,1.0
1210,AS14.33,2014-05-27,0.012704,2.256099,4.100707,4.148628,0.000000,0.0,0.000000,2.935757,3.936893,0.000000,0.000000,2.981853,0.000000,1.0,-0.600000,0.4,6.20,113.607750,1.0
1211,AS14.33,2014-05-28,0.103301,2.359948,3.643849,3.804606,0.000000,0.0,3.822943,3.363561,4.607983,0.000000,0.000000,1.810090,0.000000,1.0,0.000000,1.2,8.20,113.657683,1.0
1212,AS14.33,2014-05-29,0.169354,2.008504,4.014365,4.239440,0.000000,0.0,0.000000,2.373416,5.297718,0.662172,0.000000,0.955319,0.000000,1.0,-1.333333,1.0,7.00,118.978033,1.0


In [43]:
df.to_csv("processed_data.csv", index=False)